# 02 - Feature Engineering and USS Dimensions

Tujuan:
- Konsolidasi fitur per kelurahan
- Bentuk skor dimensi climate, infrastructure, socioeconomic
- Simpan dataset fitur untuk model comparison

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
dataRoot = Path("../data")
processedDir = dataRoot / "processed"
basePath = processedDir / "features_base.csv"

df = pd.DataFrame()
if basePath.exists():
    df = pd.read_csv(basePath)
    display(df.head(3))
else:
    print(f"Missing: {basePath}. Update basePath or create the file.")

In [ ]:
if df.empty:
    print("No data loaded.")
else:
    dimensionMap = {
        "climate": ["rainfall", "temperature", "humidity", "rainfallAnomalyZ"],
        "infrastructure": ["drainageQuality", "roadDamageRatio", "buildingDensity", "greenSpaceRatio"],
        "socioeconomic": ["povertyRate", "unemploymentRate", "healthAccessIndex", "educationIndex"]
    }

    def selectCols(df, cols):
        return [c for c in cols if c in df.columns]

    def zScore(df, cols):
        if not cols:
            return pd.DataFrame(index=df.index)
        mean = df[cols].mean()
        std = df[cols].std(ddof=0).replace(0, np.nan)
        return (df[cols] - mean) / std

    for dim, cols in dimensionMap.items():
        usedCols = selectCols(df, cols)
        dimZ = zScore(df, usedCols)
        if dimZ.empty:
            df[f"{dim}Score"] = np.nan
        else:
            df[f"{dim}Score"] = dimZ.mean(axis=1)

    scoreCols = [c for c in df.columns if c.endswith("Score")]
    display(df[scoreCols].head(3))

In [ ]:
if not df.empty:
    scoreCols = ["climateScore", "infrastructureScore", "socioeconomicScore"]
    if all(c in df.columns for c in scoreCols):
        weightMap = {"climate": 0.34, "infrastructure": 0.33, "socioeconomic": 0.33}
        df["ussScore"] = (
            weightMap["climate"] * df["climateScore"]
            + weightMap["infrastructure"] * df["infrastructureScore"]
            + weightMap["socioeconomic"] * df["socioeconomicScore"]
        )
        threshold = 70
        df["ussLabel"] = (df["ussScore"] >= threshold).astype(int)
        display(df[scoreCols + ["ussScore", "ussLabel"]].head(5))
    else:
        print("Missing score columns:", scoreCols)

In [ ]:
outputPath = processedDir / "uss_features.csv"
processedDir.mkdir(parents=True, exist_ok=True)

if not df.empty:
    df.to_csv(outputPath, index=False)
    print(f"Saved: {outputPath}")